<a href="https://colab.research.google.com/github/ElionLAB/OOP_2026_Practice/blob/main/ch_06/src/part_2/answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Environment Setup — Auto-detect Google Colab / Local
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules

# Section 9.6 needs the jsonschema library. Install it once, idempotently.
try:
    import jsonschema  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'jsonschema'])
    import jsonschema  # noqa: F401

if IN_COLAB:
    pass
else:
    print('Local environment: Make sure `conda activate oop_practice` is active.')
print('jsonschema is ready.')

# Lecture 6 — Part 2 (Slides 1–59): Strings, Bytes, Paths, Serialization & a K-NN Data-Ingestion Pipeline

**Konkuk University OOP (Python Object-Oriented Programming) — Spring 2026**

---

## Learning Objectives

1. Query and transform strings with built-in methods (`count`, `find`, `split`, `join`, `partition`).
2. Use **f-strings** — interpolation, escaped braces, inline expressions, the `=` debug suffix.
3. Drive output layout with **format specifiers** (`.2f`, `0>4`, `*^20`, `,d`) and override `__format__`.
4. Use `.format()` for **delayed-execution** templates.
5. Cross the **`str` ↔ `bytes` boundary** safely — `.encode()` / `.decode()` and `errors=` fallbacks.
6. Use `pathlib.Path` as an OOP filesystem — the overloaded `/`, `glob`, `with_suffix`, `relative_to`.
7. Serialize with **pickle**, control state via `__getstate__` / `__setstate__`, evolve schemas.
8. Serialize with **JSON**, extend `json.JSONEncoder` and use `object_hook` to round-trip custom classes.
9. Build a polymorphic **K-NN data-ingestion** pipeline: one `data_iter()` interface, four readers (CSV-dict, headerless CSV, JSON, NDJSON), plus a `jsonschema` validator — and pipe the result straight into Lecture 6.1's higher-order `partition()`.

## How this notebook is structured

Each section follows the same rhythm:

1. **Concept** — short explanation, taken directly from the slides.
2. **Full code reference** — the *goal* code block (your target).
3. **TODO** cell — fill the small blanks marked `# TODO`.
4. **Quick check** — assertions verify your solution.

The TODO blanks are intentionally tiny — usually one expression per line — so you can focus on understanding *each* line, not on writing big code from scratch.

> **Continuity with Lecture 5 and Lecture 6.1.** In Lectures 5 §7 and 6.1 §8 we hand-wrote sample data like `[{"id": "A"}, ...]` and split it with `CountingDealingPartition` / `partition(rule)`. Section 9 of this notebook plugs the missing front-end: real CSV/JSON/NDJSON readers, all hiding behind the same `data_iter()` interface, feeding samples straight into the partition machine from 6.1.

## 1. String Querying & Transformation (Slides 4–7)

> *"A string (`str`) in Python is an **immutable** sequence of Unicode characters."* — Slide 4

Two families of methods:

| Family             | Examples                                  | Returns                      |
| ------------------ | ----------------------------------------- | ---------------------------- |
| Querying (read)    | `count`, `find`, `rfind`, `startswith`    | int / bool / int (no copy)   |
| Transformation     | `upper`, `replace`, `strip`, `split`, `join`, `partition` | **new** str / list / tuple |

⚠️ Strings are **immutable** — *every* transformation method returns a NEW string object.

### 1.1 — Querying & Searching (Slide 5)

`count` counts **non-overlapping** occurrences. `find` returns the leftmost index (`-1` if absent). `rfind` searches from the right. `startswith` returns a bool.

### Full code reference (Slide 5)

```python
s = "hello world"

# Counting occurrences
print(s.count('l'))   # 3

# Finding substrings
print(s.find('o'))    # 4 (first)
print(s.rfind('o'))   # 7 (last)

# Safe prefix checking
if s.startswith("hello"):
    print("Greeting detected.")
```

In [ ]:
# TODO 1.1 — String querying

s = "hello world"

# TODO 1.1-a: count how many 'l' characters appear in s
l_count = s.count('l')

# TODO 1.1-b: find the FIRST index of 'o' in s
first_o = s.find('o')

# TODO 1.1-c: find the LAST index of 'o' in s
last_o = s.rfind('o')

# TODO 1.1-d: True iff s starts with "hello"
greets = s.startswith("hello")

print(f"count('l') = {l_count}")
print(f"find('o')  = {first_o}")
print(f"rfind('o') = {last_o}")
print(f"startswith('hello') = {greets}")


In [ ]:
# Quick check
assert l_count == 3, f"expected 3, got {l_count}"
assert first_o == 4, f"expected 4, got {first_o}"
assert last_o == 7, f"expected 7, got {last_o}"
assert greets is True, f"expected True, got {greets}"
print("OK: all four string queries match the slide.")
print("Section 1.1 passed.")

### 1.2 — Splitting and Joining (Slides 6–7)

| Method                  | Direction         | Returns           |
| ----------------------- | ----------------- | ----------------- |
| `s.split(delim)`        | str → list        | list of str       |
| `delim.join(iterable)`  | list → str        | str               |
| `s.partition(delim)`    | str → 3-tuple     | `(before, delim, after)` |

`partition` is convenient when you want to split *exactly once* on the first occurrence and keep the delimiter.

### Full code reference (Slide 7)

```python
s = "hello world, how are you"

# Split string into a list of words
word_list = s.split(' ')
print(word_list)
# ['hello', 'world,', 'how', 'are', 'you']

# Join back with a new delimiter
joined_string = '#'.join(word_list)
print(joined_string)
# 'hello#world,#how#are#you'

# Extract specific parts
print(s.partition(', '))
# ('hello world', ', ', 'how are you')
```

In [ ]:
# TODO 1.2 — split / join / partition

s = "hello world, how are you"

# TODO 1.2-a: split s on a single space character
word_list = s.split(' ')

# TODO 1.2-b: join word_list back with '#' as separator
joined_string = '#'.join(word_list)

# TODO 1.2-c: partition s on the substring ", "
parts = s.partition(', ')

print(word_list)
print(joined_string)
print(parts)


In [ ]:
# Quick check
assert word_list == ['hello', 'world,', 'how', 'are', 'you']
assert joined_string == 'hello#world,#how#are#you'
assert parts == ('hello world', ', ', 'how are you')
print("OK: split, join, and partition all match the slide.")
print("Section 1.2 passed.")

## 2. F-Strings — Modern String Formatting (Slides 8–13)

> *"Python 3.6 introduced 'f-strings' (formatted string literals). Prepend `f` to the quote, Python evaluates variables inside `{}`."* — Slide 8

F-strings replace clunky concatenation:

```python
"Hello " + name + ", you are " + str(age) + " years old."
f"Hello {name}, you are {age} years old."   # ✓ same output, no str() needed
```

### 2.1 — Basic Interpolation (Slide 9)

Variables inside `{}` are evaluated and converted to string form automatically.

### Full code reference (Slide 9)

```python
name = "Dusty"
activity = "reviewing code"

# The variables are dynamically interpolated
message = f"Hello {name}, you are currently {activity}."
print(message)
# Output: Hello Dusty, you are currently reviewing code.
```

In [ ]:
# TODO 2.1 — Basic f-string interpolation

name = "Dusty"
activity = "reviewing code"

# TODO 2.1-a: build an f-string saying  "Hello {name}, you are currently {activity}."
message = f"Hello {name}, you are currently {activity}."

print(message)


In [ ]:
# Quick check
assert message == "Hello Dusty, you are currently reviewing code."
print(f"OK: {message!r}")
print("Section 2.1 passed.")

### 2.2 — Escaped Braces: Generating Java Code (Slides 10–11)

If you need a **literal** `{` or `}` inside an f-string (e.g., emitting JSON, Java, CSS), double the brace: `{{` → `{`, `}}` → `}`.

### Full code reference (Slide 11)

```python
classname = "MyClass"
message = "hello world"

# Double braces escape the templating engine
template = f"""
public class {classname} {{
    public static void main(String[] args) {{
        System.out.println("{message}");
    }}
}}
"""
print(template)
```

In [ ]:
# TODO 2.2 — Escape literal braces inside an f-string

classname = "MyClass"
message = "hello world"

# TODO 2.2-a: build a multi-line f-string that emits a valid Java class.
#             The braces around the class body and main() body MUST appear in the OUTPUT,
#             so they need to be doubled inside the f-string.
template = f"""
public class {classname} {{
    public static void main(String[] args) {{
        System.out.println("{message}");
    }}
}}
"""

print(template)


In [ ]:
# Quick check
expected = """
public class MyClass {
    public static void main(String[] args) {
        System.out.println("hello world");
    }
}
"""
assert template == expected, "template does not match the slide's Java output"
print("OK: escaped braces produced the literal Java syntax.")
print("Section 2.2 passed.")

### 2.3 — Expressions and the `=` Debug Suffix (Slides 12–13)

> *"The curly braces in f-strings accept any valid Python expression, not just variable names."* — Slide 12

You can call methods, do arithmetic, look up dict keys, etc. The `=` suffix (Python 3.8+) prints both the expression text **and** its value — great for debugging.

### Full code reference (Slide 13)

```python
name = "python"

# Method call inside f-string
print(f"I love {name.capitalize()}!")
# 'I love Python!'

a = 5
b = 7

# The '=' suffix prints expression + result (Debug)
print(f"{a=}, {b=}, {a * b = }")
# 'a=5, b=7, a * b = 35'
```

In [ ]:
# TODO 2.3 — Inline expressions and the = debug suffix

name = "python"

# TODO 2.3-a: f-string that calls name.capitalize() inside the braces
greeting = f"I love {name.capitalize()}!"

a = 5
b = 7

# TODO 2.3-b: f-string using the `=` debug suffix for a, b, and the expression `a * b`
#             Keep the spacing exactly as the slide shows: "{a=}, {b=}, {a * b = }"
debug_line = f"{a=}, {b=}, {a * b = }"

print(greeting)
print(debug_line)


In [ ]:
# Quick check
assert greeting == "I love Python!"
assert debug_line == "a=5, b=7, a * b = 35"
print(f"OK: {greeting!r}")
print(f"OK: {debug_line!r}")
print("Section 2.3 passed.")

## 3. Format Specifiers (Slides 14–19)

> The colon `:` inside `{}` opens a **format specifier**. Full grammar (slide 16):
>
> ```
> :[fill][align][width][,][.precision][type]
> ```

| Piece          | Example      | Effect                                  |
| -------------- | ------------ | --------------------------------------- |
| `[fill][align]`| `0>`, `*^`   | pad character + `<` / `>` / `^` align  |
| `[width]`      | `10`         | minimum total width                    |
| `[,]`          | `,d`         | thousands separator                    |
| `[.precision]` | `.2f`        | digits after the decimal point         |
| `[type]`       | `f` `d` `x`  | float, int (decimal), int (hex), …     |

### 3.1 — Floating Point Precision (Slide 15)

`.2f` guarantees exactly two digits after the decimal — `0.86` instead of `0.8638`.

### Full code reference (Slide 15)

```python
subtotal = 12.34
tax = subtotal * 0.07   # 0.8638
total = subtotal + tax

# Using .2f for standard 2-decimal float formatting
receipt = f"Sub: ${subtotal:.2f} Tax: ${tax:.2f} Total: ${total:.2f}"
print(receipt)
# Output: Sub: $12.34 Tax: $0.86 Total: $13.20
```

In [ ]:
# TODO 3.1 — Floating point precision with .2f

subtotal = 12.34
tax = subtotal * 0.07     # 0.8638
total = subtotal + tax

# TODO 3.1-a: build the receipt string using f"...{x:.2f}..." for each amount.
#             Format exactly: 'Sub: $12.34 Tax: $0.86 Total: $13.20'
receipt = f"Sub: ${subtotal:.2f} Tax: ${tax:.2f} Total: ${total:.2f}"

print(receipt)


In [ ]:
# Quick check
assert receipt == "Sub: $12.34 Tax: $0.86 Total: $13.20", repr(receipt)
print(f"OK: {receipt!r}")
print("Section 3.1 passed.")

### 3.2 — Padding, Centering, and Grouping (Slide 17)

| Spec      | Example output for the input on the left          |
| --------- | ------------------------------------------------- |
| `42:0>4d` | `'0042'` — pad 0s on the LEFT to width 4          |
| `'REPORT':*^20` | `'*******REPORT*******'` — center, fill `*`  |
| `1234567890:,d` | `'1,234,567,890'` — thousands separators    |

### Full code reference (Slide 17)

```python
# Zero-padding for IDs
user_id = 42
print(f"User: {user_id:0>4d}")           # User: 0042

# Centering with a fill character
title = "REPORT"
print(f"{title:*^20}")                   # *******REPORT*******

# Thousands separators
large_num = 1234567890
print(f"Population: {large_num:,d}")     # Population: 1,234,567,890
```

In [ ]:
# TODO 3.2 — Padding, centering, grouping

user_id = 42
title = "REPORT"
large_num = 1234567890

# TODO 3.2-a: zero-pad user_id on the LEFT to width 4 as integer  →  'User: 0042'
id_line = f"User: {user_id:0>4d}"

# TODO 3.2-b: center title to width 20, fill with '*'  →  '*******REPORT*******'
title_line = f"{title:*^20}"

# TODO 3.2-c: format large_num with thousands separators as integer  →  'Population: 1,234,567,890'
pop_line = f"Population: {large_num:,d}"

print(id_line)
print(title_line)
print(pop_line)


In [ ]:
# Quick check
assert id_line    == "User: 0042"
assert title_line == "*******REPORT*******"
assert pop_line   == "Population: 1,234,567,890"
print("OK:", repr(id_line))
print("OK:", repr(title_line))
print("OK:", repr(pop_line))
print("Section 3.2 passed.")

### 3.3 — Custom `__format__` Delegation (Slides 18–19)

> *"When you pass an object into an f-string, Python calls the object's `__format__` magic method."* — Slide 18

The colon-spec after the object (`{t:F}`) is passed **verbatim** as the `format_spec` argument. Your class decides what those characters mean — domain-specific specs become possible.

### Full code reference (Slide 19)

```python
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def __format__(self, format_spec):
        if format_spec == 'F':
            val = (self.celsius * 9/5) + 32
            return f"{val:.1f}° Fahrenheit"
        return f"{self.celsius:.1f}° Celsius"


t = Temperature(25)
print(f"Today is {t:F}")   # 77.0° Fahrenheit
print(f"Today is {t}")     # 25.0° Celsius
```

In [ ]:
# TODO 3.3 — Overriding __format__

class Temperature:
    def __init__(self, celsius):
        # TODO 3.3-a: store celsius on self
        self.celsius = celsius

    def __format__(self, format_spec):
        # TODO 3.3-b: if format_spec equals the string 'F', convert to Fahrenheit and return
        #             a string formatted like "77.0° Fahrenheit"
        if format_spec == 'F':
            val = (self.celsius * 9/5) + 32
            return f"{val:.1f}° Fahrenheit"
        # TODO 3.3-c: otherwise (default branch) return celsius formatted like "25.0° Celsius"
        return f"{self.celsius:.1f}° Celsius"


In [ ]:
# Quick check
t = Temperature(25)
assert f"{t:F}" == "77.0° Fahrenheit", f"got {f'{t:F}'!r}"
assert f"{t}"   == "25.0° Celsius",    f"got {f'{t}'!r}"
print(f"OK: f\"{{t:F}}\" → {f'{t:F}'!r}")
print(f"OK: f\"{{t}}\"   → {f'{t}'!r}")

# Round-trip: 0°C → 32°F
cold = Temperature(0)
assert f"{cold:F}" == "32.0° Fahrenheit"
print("OK: 0°C formats as 32.0° Fahrenheit")
print("Section 3.3 passed.")

## 4. `.format()` for Delayed Templates (Slides 20–21)

F-strings need their variables to be **in scope when the literal is written**. When the template comes from outside (config file, DB row, email template) you don't have that — `str.format()` lets you define the template first and inject values later.

| Substitution form           | Reads                                |
| --------------------------- | ------------------------------------ |
| `"{0}"`                     | first positional argument            |
| `"{0[email]}"`              | item lookup `args[0]['email']`       |
| `"{name}"`                  | keyword argument `name=...`          |

### Full code reference (Slide 21)

```python
# The template is defined without knowing variables yet
template = "To: {0[email]}\nSubject: {0[subject]}\nBody: {body}"

message_data = {
    'email': 'user@example.com',
    'subject': 'Your Invoice',
}

# The injection happens later
final_text = template.format(message_data, body="Total: $15.00")
print(final_text)
```

In [ ]:
# TODO 4 — Delayed-execution template with str.format()

# TODO 4-a: define the template string EXACTLY as in the slide:
#           "To: {0[email]}\nSubject: {0[subject]}\nBody: {body}"
template = "To: {0[email]}\nSubject: {0[subject]}\nBody: {body}"

message_data = {
    'email': 'user@example.com',
    'subject': 'Your Invoice',
}

# TODO 4-b: call template.format(...) passing message_data as the FIRST positional arg
#           and body="Total: $15.00" as a KEYWORD arg
final_text = template.format(message_data, body="Total: $15.00")

print(final_text)


In [ ]:
# Quick check
expected = "To: user@example.com\nSubject: Your Invoice\nBody: Total: $15.00"
assert final_text == expected, repr(final_text)
print("OK: template rendered as:")
for line in final_text.splitlines():
    print("   ", line)
print("Section 4 passed.")

## 5. Bytes & Encoding (Slides 22–30)

> *"`str` lives **inside** Python (abstract Unicode code points). Bytes live **outside** (physical 8-bit integers on disk/network). Crossing either way is never automatic."* — Slide 25

```
DECODE: raw bytes ──.decode('utf-8')──▶ str
ENCODE: str ───────.encode('utf-8')──▶ raw bytes
```

### 5.1 — Unicode Code Points (Slide 23)

A Python string is a sequence of **code points** (integers identifying characters in the Unicode standard). `ord(c)` gives the integer; `:04X` formats it as a 4-digit uppercase hex (Unicode convention).

### Full code reference (Slide 23)

```python
# A standard Python string handling international text
abstract_text = "cliché 🐍"
print(abstract_text)
print(f"Length: {len(abstract_text)} characters")

# Looking under the hood at Unicode code points
for char in abstract_text:
    print(f"{char}: U+{ord(char):04X}")
```

In [ ]:
# TODO 5.1 — Inspecting code points

abstract_text = "cliché 🐍"

# TODO 5.1-a: print abstract_text directly
print(abstract_text)

# TODO 5.1-b: print f"Length: {len(abstract_text)} characters"
print(f"Length: {len(abstract_text)} characters")

# TODO 5.1-c: loop char by char and print f"{char}: U+{ord(char):04X}"
#             Hint: ord(char) returns the code point as an int; :04X is 4-digit uppercase hex.
for char in abstract_text:
    print(f"{char}: U+{ord(char):04X}")


In [ ]:
# Quick check — capture the printed lines and compare.
import io, contextlib

abstract_text = "cliché 🐍"
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print(abstract_text)
    print(f"Length: {len(abstract_text)} characters")
    for char in abstract_text:
        print(f"{char}: U+{ord(char):04X}")

lines = buf.getvalue().splitlines()
assert lines[0] == "cliché 🐍"
assert lines[1] == "Length: 8 characters"      # 7 visible chars + the 🐍 (1 code point)
# Spot-check a couple of code points
assert "c: U+0063" in lines
assert "é: U+00E9" in lines
assert "🐍: U+1F40D" in lines
for line in lines:
    print(line)
print("Section 5.1 passed.")

### 5.2 — Decoding Bytes → str (Slide 26)

The `b''` prefix marks a **bytes** literal. `\xc3\xa9` is the UTF-8 byte sequence for `é` (U+00E9). Wrong charset = `UnicodeDecodeError` or *mojibake*.

### Full code reference (Slide 26)

```python
# Raw bytes from file/network (Notice the 'b' prefix)
raw_data = b'clich\xc3\xa9'
print(type(raw_data))  # <class 'bytes'>

# Decode to a string using UTF-8
text = raw_data.decode('utf-8')
print(type(text))      # <class 'str'>
print(text)            # cliché
```

In [ ]:
# TODO 5.2 — Decode bytes to str

# These raw bytes came from somewhere outside Python (file, socket, ...)
raw_data = b'clich\xc3\xa9'

# TODO 5.2-a: decode raw_data using the UTF-8 codec
text = raw_data.decode('utf-8')

print(type(raw_data))
print(type(text))
print(text)


In [ ]:
# Quick check
assert isinstance(raw_data, bytes)
assert isinstance(text, str)
assert text == "cliché"
print(f"OK: bytes {raw_data!r} → str {text!r}")
print("Section 5.2 passed.")

### 5.3 — Encoding Fallback Strategies (Slides 27–28)

What happens when the target codec can't represent a character?

| `errors=`              | Behavior on un-encodable char                          |
| ---------------------- | ------------------------------------------------------ |
| `'strict'` *(default)* | raise `UnicodeEncodeError`                             |
| `'ignore'`             | silently drop the character                            |
| `'replace'`            | substitute `?`                                         |
| `'xmlcharrefreplace'`  | substitute an XML char reference: `é` → `&#233;`       |

### Full code reference (Slide 28)

```python
text = "cliché"

# replace
print(text.encode("ascii", errors="replace"))
# b'clich?'

# ignore
print(text.encode("ascii", errors="ignore"))
# b'clich'

# xmlcharrefreplace
print(text.encode("ascii", errors="xmlcharrefreplace"))
# b'clich&#233;'
```

In [ ]:
# TODO 5.3 — Three fallback strategies

text = "cliché"

# TODO 5.3-a: encode to ASCII with errors='replace'   → b'clich?'
b_replace = text.encode("ascii", errors="replace")

# TODO 5.3-b: encode to ASCII with errors='ignore'    → b'clich'
b_ignore  = text.encode("ascii", errors="ignore")

# TODO 5.3-c: encode to ASCII with errors='xmlcharrefreplace'  → b'clich&#233;'
b_xml     = text.encode("ascii", errors="xmlcharrefreplace")

print(b_replace)
print(b_ignore)
print(b_xml)


In [ ]:
# Quick check
assert b_replace == b'clich?'
assert b_ignore  == b'clich'
assert b_xml     == b'clich&#233;'
print(f"OK: replace            → {b_replace}")
print(f"OK: ignore             → {b_ignore}")
print(f"OK: xmlcharrefreplace  → {b_xml}")
print("Section 5.3 passed.")

### 5.4 — `bytearray` — Mutable Binary Buffer (Slides 29–30)

`bytes` is immutable (like `str`). For in-place edits — network packets, image streams — use `bytearray`. It behaves like a list of integers `0..255`.

### Full code reference (Slide 30)

```python
ba = bytearray(b"abcdefgh")
print(ba)            # bytearray(b'abcdefgh')

# Modify in-place using index assignment
ba[1] = ord('g')     # Replace with 'g'
ba[2] = 68           # Replace with 'D' (ASCII 68)
print(ba)            # bytearray(b'agDdefgh')

# It acts like a list of ints
ba.append(122)       # Append 'z'
print(ba)            # bytearray(b'agDdefghz')
```

In [ ]:
# TODO 5.4 — Mutating bytes in place

ba = bytearray(b"abcdefgh")
print(ba)

# TODO 5.4-a: assign the integer ord('g') to position 1
ba[1] = ord('g')

# TODO 5.4-b: assign the integer 68 (ASCII 'D') to position 2
ba[2] = 68

print(ba)

# TODO 5.4-c: append the integer 122 (ASCII 'z')
ba.append(122)

print(ba)


In [ ]:
# Quick check
assert ba == bytearray(b'agDdefghz'), f"got {ba}"
print(f"OK: in-place edits produced {ba}")
print("Section 5.4 passed.")

## 6. `pathlib` — OOP Filesystem Paths (Slides 31–34)

> *"`pathlib` provides an object-oriented interface to the filesystem, replacing the older `os.path` string-manipulation functions."* — Slide 31

Key win: the `/` operator is **overloaded** on `Path` objects, so cross-platform path joining reads like a directory tree.

### 6.1 — Path Construction with `/` (Slide 32)

`Path.cwd()` returns the **C**urrent **W**orking **D**irectory as a `Path`. The overloaded `/` joins another segment, and `.parent` navigates up.

### Full code reference (Slide 32)

```python
from pathlib import Path

# Get the Current Working Directory (CWD)
base = Path.cwd()

# The '/' operator is overloaded by Path objects!
chapter_dir = base.parent / "ch_02" / "data"
print(chapter_dir)
# /home/user/projects/ch_02/data       (on Unix)
# C:\Users\user\projects\ch_02\data  (on Windows)
```

In [ ]:
# TODO 6.1 — Compose a Path with the overloaded /
from pathlib import Path

# TODO 6.1-a: get the current working directory as a Path
base = Path.cwd()

# TODO 6.1-b: build base.parent / "ch_02" / "data" using the / operator
chapter_dir = base.parent / "ch_02" / "data"

print(chapter_dir)


In [ ]:
# Quick check — assert by structure, not by absolute path (works on any machine).
assert isinstance(base, Path)
assert isinstance(chapter_dir, Path)
# the last two segments must be ch_02/data
assert chapter_dir.parts[-2:] == ("ch_02", "data"), chapter_dir.parts
# chapter_dir == base.parent / "ch_02" / "data"
assert chapter_dir == base.parent / "ch_02" / "data"
print(f"OK: base        = {base}")
print(f"OK: chapter_dir = {chapter_dir}")
print("Section 6.1 passed.")

### 6.2 — `glob` + `with_suffix` + `relative_to` (Slides 33–34)

| Method                | Purpose                                            |
| --------------------- | -------------------------------------------------- |
| `p.glob('**/*.txt')`  | recursively yield matching `Path`s                 |
| `p.with_suffix('.bak')` | swap the extension                                |
| `p.relative_to(base)` | strip a leading path prefix                        |

For the TODO below we'll create a temporary directory tree, so the example is self-contained and runnable.

### Full code reference (Slide 34)

```python
from pathlib import Path

base_dir = Path("/var/log")

# Recursively find all text files
for log_file in base_dir.glob("**/*.txt"):
    # Change the file extension dynamically
    backup_path = log_file.with_suffix(".bak")
    # Calculate the path relative to the base directory
    relative = log_file.relative_to(base_dir)
    print(f"Backing up: {relative} -> {backup_path.name}")
```

In [ ]:
# Setup — build a tiny temp tree we can really walk
import tempfile, shutil
from pathlib import Path

tmp_root = Path(tempfile.mkdtemp(prefix="oop_pathlib_"))
(tmp_root / "sub").mkdir()
(tmp_root / "a.txt").write_text("a")
(tmp_root / "sub" / "b.txt").write_text("b")
(tmp_root / "sub" / "c.log").write_text("c")   # NOT a .txt — should be skipped
print(f"Temp tree built at: {tmp_root}")

In [ ]:
# TODO 6.2 — Walk, transform suffix, compute relative path
base_dir = tmp_root  # use our temp tree (the slide used /var/log)

results = []  # collect (relative, backup_name) tuples for the quick check

# TODO 6.2-a: recursively find all *.txt files under base_dir
for log_file in base_dir.glob("**/*.txt"):
    # TODO 6.2-b: derive a path with the suffix swapped from .txt to .bak
    backup_path = log_file.with_suffix(".bak")
    # TODO 6.2-c: compute the path of log_file relative to base_dir
    relative = log_file.relative_to(base_dir)
    results.append((str(relative), backup_path.name))
    print(f"Backing up: {relative} -> {backup_path.name}")


In [ ]:
# Quick check
results_sorted = sorted(results)
expected_sorted = sorted([
    ("a.txt",                str(Path("a.bak").name)),
    (str(Path("sub") / "b.txt"), "b.bak"),
])
assert results_sorted == expected_sorted, f"got {results_sorted}"
print("OK: glob('**/*.txt') found exactly the two .txt files, .log was skipped.")
print("OK: with_suffix and relative_to produced the expected pairs.")

# Cleanup the temp tree
shutil.rmtree(tmp_root)
print("Section 6.2 passed.")

## 7. Pickle — Native Python Serialization (Slides 35–41)

> *"Pickle converts a dynamic Python object's state into a flat sequence of bytes for storage or network transfer. Like freeze-drying."* — Slide 35

| Direction         | Function       | File mode |
| ----------------- | -------------- | --------- |
| Serialize (dump)  | `pickle.dump`  | `'wb'`    |
| Deserialize (load)| `pickle.load`  | `'rb'`    |

⚠️ **Security** — *never* unpickle untrusted data. Pickle can execute arbitrary code on load. For external data exchange, use JSON (next section).

### 7.1 — Basic `dump` / `load` Round-Trip (Slide 36)

The reloaded object **equals** the original (`==`) but is a **different object** (`is not`) — pickle reconstructs a fresh copy.

### Full code reference (Slide 36)

```python
import pickle

my_data = ["hello", 42, {"key": "value"}]

# 1. Serialize (Dump) to a binary file
with open("saved_data.pkl", "wb") as file:
    pickle.dump(my_data, file)

# 2. Deserialize (Load) from a binary file
with open("saved_data.pkl", "rb") as file:
    restored_data = pickle.load(file)

print(restored_data == my_data)   # True
print(restored_data is my_data)   # False (It's a new object!)
```

In [ ]:
# TODO 7.1 — Pickle dump / load round-trip
import pickle

my_data = ["hello", 42, {"key": "value"}]

# TODO 7.1-a: open 'saved_data.pkl' in BINARY WRITE mode and dump my_data into it
with open("saved_data.pkl", "wb") as file:
    pickle.dump(my_data, file)

# TODO 7.1-b: open 'saved_data.pkl' in BINARY READ mode and load it into restored_data
with open("saved_data.pkl", "rb") as file:
    restored_data = pickle.load(file)

print(restored_data == my_data)
print(restored_data is my_data)


In [ ]:
# Quick check
assert restored_data == my_data, "values must round-trip equal"
assert restored_data is not my_data, "but pickle returns a NEW object"
assert restored_data[2] is not my_data[2], "nested dict is also a fresh copy"
print(f"OK: restored_data == my_data   → True")
print(f"OK: restored_data is not my_data → True (fresh object)")

# Tidy up the pickle file
from pathlib import Path
Path("saved_data.pkl").unlink(missing_ok=True)
print("Section 7.1 passed.")

### 7.2 — Controlling State with `__getstate__` / `__setstate__` (Slide 38)

Some attributes are **unpicklable**: open sockets, file handles, threads, DB connections. The fix:

1. `__getstate__` returns the **picklable** subset (a dict by convention).
2. `__setstate__` restores those, then **rebuilds** the ephemeral pieces.

> The original slide-38 example used a placeholder `open_network_socket(url)`. We swap it for `time.time()` — also ephemeral, but actually runnable. The *pattern* is identical.

### Full code reference (Slide 38, adapted for runnability)

```python
import time

class URLReader:
    def __init__(self, url):
        self.url = url
        # ⚠️ Ephemeral, machine/process-specific — must NOT be pickled.
        # Slide 38 used open_network_socket(url); we use time.time() to keep
        # the runnable example faithful to the same pattern.
        self.session_start = time.time()

    def __getstate__(self):
        # Return a dictionary of ONLY the state to save
        return {"url": self.url}

    def __setstate__(self, state):
        # Rebuild the object and re-establish ephemeral state
        self.url = state["url"]
        self.session_start = time.time()
```

In [ ]:
# TODO 7.2 — URLReader with __getstate__ / __setstate__
import time

class URLReader:
    def __init__(self, url):
        self.url = url
        # Ephemeral state — slide 38 used open_network_socket(url); time.time()
        # plays the same role: a value tied to "now" that we don't want serialized.
        self.session_start = time.time()

    def __getstate__(self):
        # TODO 7.2-a: return a dict containing ONLY the picklable url
        return {"url": self.url}

    def __setstate__(self, state):
        # TODO 7.2-b: restore self.url from state
        self.url = state["url"]
        # TODO 7.2-c: rebuild the ephemeral session_start with a fresh time.time()
        self.session_start = time.time()


In [ ]:
# Quick check
import pickle, time

original = URLReader("https://example.com/data.csv")
t0 = original.session_start

# A small sleep so the restored session_start is measurably newer.
time.sleep(0.01)

blob = pickle.dumps(original)
restored = pickle.loads(blob)

# url survives the round-trip
assert restored.url == original.url, restored.url
# session_start was rebuilt from scratch — it is NEWER than the original
assert restored.session_start > t0, "ephemeral state should be freshly created on load"
print(f"OK: url survived     → {restored.url!r}")
print(f"OK: session_start regenerated   (original={t0:.4f}, restored={restored.session_start:.4f})")

# __getstate__ truly omits the ephemeral piece
assert original.__getstate__() == {"url": "https://example.com/data.csv"}
print("OK: __getstate__ returned only the picklable subset.")
print("Section 7.2 passed.")

### 7.3 — Schema Evolution with `__setstate__` (Slide 40)

When a class adds a new field later (V2), old pickle files (V1) won't contain it. `__setstate__` is the migration hook: fill in defaults, normalize the version stamp.

### Full code reference (Slide 40)

```python
class Contact:
    def __init__(self, name, age, email=""):
        self.name = name
        self.age = age
        self.email = email      # Added in V2
        self.version = 2

    def __setstate__(self, state):
        self.__dict__.update(state)
        # V1 → V2 migration: fill missing fields, stamp current version
        if 'email' not in state:
            self.email = ""     # default for old data
        self.version = 2        # normalize to current schema
```

In [ ]:
# TODO 7.3 — Contact class with schema-evolution __setstate__

class Contact:
    def __init__(self, name, age, email=""):
        # TODO 7.3-a: store name, age, email; stamp self.version = 2
        self.name = name
        self.age = age
        self.email = email
        self.version = 2

    def __setstate__(self, state):
        # TODO 7.3-b: bulk-update self.__dict__ from the incoming state dict
        self.__dict__.update(state)
        # TODO 7.3-c: if 'email' was missing from the V1 state, set it to ""
        if 'email' not in state:
            self.email = ""
        # TODO 7.3-d: normalize self.version to 2 regardless of what state had
        self.version = 2


In [ ]:
# Quick check — simulate a V1 pickle (no email, version=1) being loaded by V2 code.
import pickle

# Build a V1-style state dict by hand (this is what the old pickle would have stored).
v1_state = {"name": "Alice", "age": 30, "version": 1}

# Use __new__ to get a Contact instance WITHOUT running __init__ (pickle does the same).
old = Contact.__new__(Contact)
old.__setstate__(v1_state)

assert old.name == "Alice"
assert old.age == 30
assert old.email == "", f"V1 pickle should be filled with default email='', got {old.email!r}"
assert old.version == 2, f"version should be normalized to 2, got {old.version}"
print(f"OK: V1 pickle migrated → name={old.name!r}, age={old.age}, email={old.email!r}, version={old.version}")

# A normal V2 pickle round-trip should also be fine.
fresh = Contact("Bob", 25, "bob@example.com")
loaded = pickle.loads(pickle.dumps(fresh))
assert loaded.name == "Bob" and loaded.email == "bob@example.com" and loaded.version == 2
print("OK: V2 round-trip preserves all fields.")
print("Section 7.3 passed.")

## 8. JSON — Cross-Language Serialization (Slides 42–44)

| Python                        | JSON     |
| ----------------------------- | -------- |
| `dict`                        | object   |
| `list`                        | array    |
| `str`                         | string   |
| `int` / `float`               | number   |
| `True` / `False`              | true / false |
| `None`                        | null     |

JSON is the safe choice for **anything that leaves your trust boundary**.

### 8.1 — `json.dumps` / `json.loads` (Slide 42)

`dumps` (`dump-s`tring) returns a `str`; `loads` parses one back.

### Full code reference (Slide 42)

```python
import json

data = {"name": "Alice", "scores": [1, 2, 3], "active": True}

# dumps = Dump String
s = json.dumps(data, indent=2)
print(s)

# loads = Load String
parsed = json.loads(s)
```

In [ ]:
# TODO 8.1 — JSON dumps / loads
import json

data = {"name": "Alice", "scores": [1, 2, 3], "active": True}

# TODO 8.1-a: serialize data to a JSON string with indent=2
s = json.dumps(data, indent=2)

# TODO 8.1-b: parse s back into a Python object
parsed = json.loads(s)

print(s)
print("---")
print(parsed)


In [ ]:
# Quick check
assert isinstance(s, str)
assert "Alice" in s and "scores" in s
# Indented form contains newlines + two-space indentation
assert "\n  \"" in s, "indent=2 should add 2-space indentation"
assert parsed == data, "round-trip equality must hold"
assert parsed is not data
print("OK: dumps produced indented JSON")
print("OK: loads round-tripped to an equal-but-fresh dict")
print("Section 8.1 passed.")

### 8.2 — Custom Classes via `JSONEncoder` + `object_hook` (Slides 43–44)

JSON doesn't natively know your `Contact` class. The pattern (slide 43):

1. Subclass `json.JSONEncoder`, override `default(self, obj)` to convert your class to a dict.
2. Tag the dict with a sentinel key like `"__class__": "Contact"` so we can recognize it on the way back.
3. On load, pass `object_hook=decode_contact` to `json.loads` — it runs on every parsed dict and rebuilds the object when the sentinel is present.

### Full code reference (Slide 44)

```python
import json

class ContactEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Contact):
            return {"__class__": "Contact",
                    "name": obj.name, "age": obj.age, "email": obj.email}
        return super().default(obj)


def decode_contact(dic):
    if dic.get("__class__") == "Contact":
        return Contact(dic["name"], dic["age"], dic["email"])
    return dic


c = Contact("John Smith", 30, "john@example.com")
text = json.dumps(c, cls=ContactEncoder)
new_c = json.loads(text, object_hook=decode_contact)
```

In [ ]:
# TODO 8.2 — Custom JSONEncoder + object_hook for Contact
import json


class ContactEncoder(json.JSONEncoder):
    def default(self, obj):
        # TODO 8.2-a: if obj is a Contact, return a tagged dict:
        #             {"__class__": "Contact", "name": ..., "age": ..., "email": ...}
        if isinstance(obj, Contact):
            return {"__class__": "Contact",
                    "name": obj.name, "age": obj.age, "email": obj.email}
        # TODO 8.2-b: otherwise fall back to the parent's default
        return super().default(obj)


def decode_contact(dic):
    # TODO 8.2-c: if dic has "__class__" == "Contact", rebuild a Contact and return it
    if dic.get("__class__") == "Contact":
        return Contact(dic["name"], dic["age"], dic["email"])
    return dic


In [ ]:
# Quick check
import json

c = Contact("John Smith", 30, "john@example.com")

text = json.dumps(c, cls=ContactEncoder)
assert '"__class__": "Contact"' in text, "encoder must tag the dict with __class__"
assert '"John Smith"' in text
print(f"OK: encoded → {text}")

new_c = json.loads(text, object_hook=decode_contact)
assert isinstance(new_c, Contact)
assert (new_c.name, new_c.age, new_c.email) == ("John Smith", 30, "john@example.com")
print(f"OK: decoded back to Contact: name={new_c.name!r}, age={new_c.age}, email={new_c.email!r}")

# Built-in types still round-trip cleanly (encoder.default only intercepts Contact)
mixed = json.dumps([c, {"plain": True}], cls=ContactEncoder)
mixed_back = json.loads(mixed, object_hook=decode_contact)
assert isinstance(mixed_back[0], Contact)
assert mixed_back[1] == {"plain": True}
print("OK: mixed list of Contact + plain dict round-trips correctly")
print("Section 8.2 passed.")

## 9. Case Study — K-NN Data Ingestion (Slides 45–58)

### Where this fits in the trilogy

| Lecture       | Layer        | What it produced                                              |
| ------------- | ------------ | ------------------------------------------------------------- |
| 5 §7          | Split (OOP)  | `CountingDealingPartition` — a *class* that dispatches items   |
| 6.1 §8        | Split (func) | `partition(samples, rule)` — a *higher-order function*         |
| **6.2 §9**    | **Ingest**   | **Readers that produce `samples` from CSV / JSON / NDJSON**   |

Until now we wrote `[{"id": "A"}, ...]` by hand. This section builds the missing front-end.

### Slide-46 design

```
CSVIrisReader       ──┐
HeaderlessCSVReader ──┤   data_iter()      yields Sample
JSONIrisReader      ──┼──▶ shared ABC ───▶ objects, lazily
NDJSONIrisReader    ──┘
```

One interface, four implementations — *polymorphism through ABCs*, exactly the Open/Closed pattern from Lecture 5 Part 1.

### 9.0 — Setup: write small Iris-style data files

We'll generate four equivalent representations of the same 10 samples so each reader can be exercised against real files. All four readers must yield the **same** dict rows.

In [ ]:
# Setup — create iris.csv, iris_headerless.csv, iris.json, iris.ndjson
import json as _json
from pathlib import Path

HEADERS = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

IRIS_RAW = [
    (5.1, 3.5, 1.4, 0.2, "Iris-setosa"),
    (4.9, 3.0, 1.4, 0.2, "Iris-setosa"),
    (4.7, 3.2, 1.3, 0.2, "Iris-setosa"),
    (4.6, 3.1, 1.5, 0.2, "Iris-setosa"),
    (5.0, 3.6, 1.4, 0.2, "Iris-setosa"),
    (7.0, 3.2, 4.7, 1.4, "Iris-versicolor"),
    (6.4, 3.2, 4.5, 1.5, "Iris-versicolor"),
    (6.9, 3.1, 4.9, 1.5, "Iris-versicolor"),
    (6.3, 3.3, 6.0, 2.5, "Iris-virginica"),
    (5.8, 2.7, 5.1, 1.9, "Iris-virginica"),
]

# Build dicts once for the JSON/NDJSON writers
IRIS_DICTS = [dict(zip(HEADERS, row)) for row in IRIS_RAW]

# 1) CSV WITH header
Path("iris.csv").write_text(
    ",".join(HEADERS) + "\n" +
    "\n".join(",".join(str(v) for v in row) for row in IRIS_RAW) + "\n",
    encoding="utf-8",
)

# 2) CSV WITHOUT header (data rows only)
Path("iris_headerless.csv").write_text(
    "\n".join(",".join(str(v) for v in row) for row in IRIS_RAW) + "\n",
    encoding="utf-8",
)

# 3) Standard JSON — a single array
Path("iris.json").write_text(_json.dumps(IRIS_DICTS, indent=2), encoding="utf-8")

# 4) NDJSON — one record per line
Path("iris.ndjson").write_text(
    "\n".join(_json.dumps(d) for d in IRIS_DICTS) + "\n",
    encoding="utf-8",
)

for p in ("iris.csv", "iris_headerless.csv", "iris.json", "iris.ndjson"):
    print(f"  {p}  ({Path(p).stat().st_size} bytes)")
print(f"\n{len(IRIS_DICTS)} samples written across 4 formats.")

### 9.1 — Abstract Base `IrisReader` (Slide 46)

The contract: every concrete reader has a `data_iter()` method that yields one sample dict at a time. Marking it `@abstractmethod` makes the ABC refuse to instantiate any subclass that forgets to implement it (Lecture 5 §6 territory).

### Full code reference (derived from Slide 46's interface)

```python
import abc
from pathlib import Path

class IrisReader(abc.ABC):
    def __init__(self, source: Path):
        self.source = source

    @abc.abstractmethod
    def data_iter(self):
        ...
```

In [ ]:
# TODO 9.1 — Abstract base IrisReader
import abc
from pathlib import Path


class IrisReader(abc.ABC):
    def __init__(self, source: Path):
        # TODO 9.1-a: store source on self
        self.source = source

    # TODO 9.1-b: decorate data_iter as an abstractmethod (body can be `...` or `pass`)
    @abc.abstractmethod
    def data_iter(self):
        ...


In [ ]:
# Quick check
assert "data_iter" in IrisReader.__abstractmethods__, \
    "data_iter must be registered as an abstract method"
try:
    IrisReader(Path("iris.csv"))
except TypeError as e:
    print(f"OK: cannot instantiate the abstract base directly — {e}")
else:
    raise AssertionError("IrisReader() should have raised TypeError")
print("Section 9.1 passed.")

### 9.2 — `CSVIrisReader` with `csv.DictReader` (Slide 49)

> **Before** — `value = float(row[5])` … fragile, column-order-dependent.
> **After** — `value = float(row["petal_length"])` … robust.

`csv.DictReader` automatically uses the first row as field names and yields dicts. `yield from reader` re-exposes them through our generator.

### Full code reference (Slide 49)

```python
import csv
from pathlib import Path

class CSVIrisReader(IrisReader):
    def data_iter(self):
        with self.source.open(newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            yield from reader
```

In [ ]:
# TODO 9.2 — CSVIrisReader
import csv


class CSVIrisReader(IrisReader):
    def data_iter(self):
        # TODO 9.2-a: open self.source with newline="" and encoding="utf-8"
        with self.source.open(newline="", encoding="utf-8") as f:
            # TODO 9.2-b: build a csv.DictReader on f
            reader = csv.DictReader(f)
            # TODO 9.2-c: yield each row from the reader
            yield from reader


In [ ]:
# Quick check
reader = CSVIrisReader(Path("iris.csv"))
rows = list(reader.data_iter())
assert len(rows) == 10
assert rows[0] == {
    "sepal_length": "5.1", "sepal_width": "3.5",
    "petal_length": "1.4", "petal_width": "0.2",
    "species": "Iris-setosa",
}, rows[0]
# csv module reads ALL values as strings — that's by design (slide 47).
assert isinstance(rows[0]["sepal_length"], str)
print(f"OK: CSVIrisReader yielded {len(rows)} dict rows")
print(f"OK: first row = {rows[0]}")
print("Section 9.2 passed.")

### 9.3 — `HeaderlessCSVReader` (Slide 51)

When the file has **no** header row, pass an explicit `fieldnames=` list to `DictReader`. Now we get the robustness of named keys even though the file itself is just bare data.

### Full code reference (Slide 51)

```python
class HeaderlessCSVReader(IrisReader):
    header = [
        "sepal_length", "sepal_width",
        "petal_length", "petal_width", "species",
    ]

    def data_iter(self):
        with self.source.open(newline="", encoding="utf-8") as source_file:
            # Provide the headers manually
            reader = csv.DictReader(source_file, fieldnames=self.header)
            yield from reader
```

In [ ]:
# TODO 9.3 — HeaderlessCSVReader

class HeaderlessCSVReader(IrisReader):
    header = [
        "sepal_length", "sepal_width",
        "petal_length", "petal_width", "species",
    ]

    def data_iter(self):
        with self.source.open(newline="", encoding="utf-8") as source_file:
            # TODO 9.3-a: build a csv.DictReader, passing fieldnames=self.header
            reader = csv.DictReader(source_file, fieldnames=self.header)
            # TODO 9.3-b: yield each row
            yield from reader


In [ ]:
# Quick check
reader = HeaderlessCSVReader(Path("iris_headerless.csv"))
rows = list(reader.data_iter())
assert len(rows) == 10, f"expected 10 rows, got {len(rows)}"
assert list(rows[0].keys()) == HeaderlessCSVReader.header
assert rows[-1]["species"] == "Iris-virginica"
print(f"OK: HeaderlessCSVReader yielded {len(rows)} rows using explicit field names")
print(f"OK: last row = {rows[-1]}")
print("Section 9.3 passed.")

### 9.4 — `JSONIrisReader` — Whole-File Load (Slides 52–53)

Standard JSON requires a single root (here, a list). `json.load(f)` reads the *entire* file into RAM, then `yield from sample_list` re-streams the items.

> Trade-off (slide 52): clean, but the whole array must fit in memory at once.

### Full code reference (Slide 53)

```python
import json

class JSONIrisReader(IrisReader):
    def data_iter(self):
        with self.source.open(encoding="utf-8") as source_file:
            # Reads the ENTIRE file into RAM at once
            sample_list = json.load(source_file)
        # Yields items one by one from the loaded list
        yield from sample_list
```

In [ ]:
# TODO 9.4 — JSONIrisReader (full-file load)
import json


class JSONIrisReader(IrisReader):
    def data_iter(self):
        with self.source.open(encoding="utf-8") as source_file:
            # TODO 9.4-a: load the entire JSON document into sample_list
            sample_list = json.load(source_file)
        # TODO 9.4-b: yield each item from sample_list
        yield from sample_list


In [ ]:
# Quick check
reader = JSONIrisReader(Path("iris.json"))
rows = list(reader.data_iter())
assert len(rows) == 10
# JSON preserves real numeric types (unlike CSV)
assert rows[0]["sepal_length"] == 5.1
assert isinstance(rows[0]["sepal_length"], float)
assert rows[0]["species"] == "Iris-setosa"
print(f"OK: JSONIrisReader yielded {len(rows)} rows with NATIVE numeric types")
print(f"OK: rows[0]['sepal_length'] = {rows[0]['sepal_length']}  (type {type(rows[0]['sepal_length']).__name__})")
print("Section 9.4 passed.")

### 9.5 — `NDJSONIrisReader` — Streaming Per Line (Slides 54–55)

NDJSON puts **one JSON object per line**, no enclosing `[ ]`. Parsing line-by-line keeps memory at O(1) regardless of file size — exactly what you need for multi-GB logs.

### Full code reference (Slide 55)

```python
class NDJSONIrisReader(IrisReader):
    def data_iter(self):
        with self.source.open(encoding="utf-8") as source_file:
            # Iterate lazily over file lines
            for line in source_file:
                # Parse one object at a time
                sample = json.loads(line)
                yield sample
```

In [ ]:
# TODO 9.5 — NDJSONIrisReader (streaming)

class NDJSONIrisReader(IrisReader):
    def data_iter(self):
        with self.source.open(encoding="utf-8") as source_file:
            # TODO 9.5-a: iterate lazily over each line of source_file
            for line in source_file:
                # TODO 9.5-b: parse a single JSON object from the line
                sample = json.loads(line)
                # TODO 9.5-c: yield the parsed sample
                yield sample


In [ ]:
# Quick check
reader = NDJSONIrisReader(Path("iris.ndjson"))
rows = list(reader.data_iter())
assert len(rows) == 10
assert rows[0]["sepal_length"] == 5.1     # numeric types preserved like JSON
assert rows[0]["species"] == "Iris-setosa"

# All four readers should agree on the data CONTENT (ignoring CSV's string-typing).
# Compare via species strings — every reader agrees on those.
species_csv     = [r["species"] for r in CSVIrisReader(Path("iris.csv")).data_iter()]
species_jsonarr = [r["species"] for r in JSONIrisReader(Path("iris.json")).data_iter()]
species_ndjson  = [r["species"] for r in NDJSONIrisReader(Path("iris.ndjson")).data_iter()]
species_hless   = [r["species"] for r in HeaderlessCSVReader(Path("iris_headerless.csv")).data_iter()]
assert species_csv == species_jsonarr == species_ndjson == species_hless
print(f"OK: NDJSONIrisReader yielded {len(rows)} streamed rows")
print(f"OK: all four readers produced the SAME species sequence:")
print(f"    {species_csv}")
print("Section 9.5 passed.")

### 9.6 — `ValidatingNDJSONReader` with `jsonschema` (Slides 56–58)

> *"Just because `json.loads()` succeeds doesn't mean the data has the correct keys or types."* — Slide 56

A `jsonschema` validator rejects rows missing required fields, with wrong types, or with extra keys (`additionalProperties: False`). Invalid rows are skipped (with a printed warning); valid ones flow on.

### Full code reference (Slides 57–58)

```python
import jsonschema

iris_schema = {
    "type": "object",
    "required": ["sepal_length", "sepal_width",
                 "petal_length", "petal_width", "species"],
    "properties": {
        "sepal_length": {"type": "number", "minimum": 0},
        "sepal_width":  {"type": "number", "minimum": 0},
        "petal_length": {"type": "number", "minimum": 0},
        "petal_width":  {"type": "number", "minimum": 0},
        "species":      {"type": "string"},
    },
    "additionalProperties": False,
}


class ValidatingNDJSONReader(IrisReader):
    def __init__(self, source: Path, schema: dict):
        super().__init__(source)
        self.validator = jsonschema.Draft7Validator(schema)

    def data_iter(self):
        with self.source.open(encoding="utf-8") as file:
            for line in file:
                sample = json.loads(line)
                errs = list(self.validator.iter_errors(sample))
                if not errs:
                    yield sample
                else:
                    print(f"Invalid: {errs[0].message}")
```

In [ ]:
# Build a small NDJSON file with deliberate bad rows to exercise the validator
from pathlib import Path

bad_lines = [
    '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2, "species": "Iris-setosa"}',   # valid
    '{"sepal_length": -1.0, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2, "species": "Iris-setosa"}',  # negative violates minimum: 0
    '{"sepal_length": 4.9, "sepal_width": 3.0, "petal_length": 1.4, "petal_width": 0.2}',                              # missing species
    '{"sepal_length": 7.0, "sepal_width": 3.2, "petal_length": 4.7, "petal_width": 1.4, "species": "Iris-versicolor", "extra": "bad"}',  # additionalProperties violation
    '{"sepal_length": 5.8, "sepal_width": 2.7, "petal_length": 5.1, "petal_width": 1.9, "species": "Iris-virginica"}',  # valid
]
Path("iris_mixed.ndjson").write_text("\n".join(bad_lines) + "\n", encoding="utf-8")
print("Wrote iris_mixed.ndjson  (5 lines: 2 valid, 3 invalid)")

In [ ]:
# TODO 9.6 — Validating reader
import jsonschema
import json

iris_schema = {
    "type": "object",
    "required": ["sepal_length", "sepal_width",
                 "petal_length", "petal_width", "species"],
    "properties": {
        "sepal_length": {"type": "number", "minimum": 0},
        "sepal_width":  {"type": "number", "minimum": 0},
        "petal_length": {"type": "number", "minimum": 0},
        "petal_width":  {"type": "number", "minimum": 0},
        "species":      {"type": "string"},
    },
    "additionalProperties": False,
}


class ValidatingNDJSONReader(IrisReader):
    def __init__(self, source: Path, schema: dict):
        # TODO 9.6-a: call super().__init__(source) so self.source gets set
        super().__init__(source)
        # TODO 9.6-b: build a Draft7Validator from the schema and store on self.validator
        self.validator = jsonschema.Draft7Validator(schema)

    def data_iter(self):
        with self.source.open(encoding="utf-8") as file:
            for line in file:
                # TODO 9.6-c: parse one JSON object from the line
                sample = json.loads(line)
                # TODO 9.6-d: collect errors as a list using self.validator.iter_errors(sample)
                errs = list(self.validator.iter_errors(sample))
                if not errs:
                    # TODO 9.6-e: yield the valid sample
                    yield sample
                else:
                    print(f"Invalid: {errs[0].message}")


In [ ]:
# Quick check
reader = ValidatingNDJSONReader(Path("iris_mixed.ndjson"), iris_schema)
valid_rows = list(reader.data_iter())
assert len(valid_rows) == 2, f"only 2 of the 5 rows are valid, got {len(valid_rows)}"
assert valid_rows[0]["species"] == "Iris-setosa"
assert valid_rows[1]["species"] == "Iris-virginica"
print(f"OK: ValidatingNDJSONReader kept {len(valid_rows)} valid rows, rejected the 3 bad ones")
print("Section 9.6 passed.")

### 9.7 — Bridge: Ingest → Partition (Continuity with Lecture 6.1)

The whole point of the shared `data_iter()` interface: downstream code consumes a **stream of samples** and doesn't care where they came from. We can pipe any reader into Lecture 6.1's `partition`:

```python
samples = list(reader.data_iter())             # 6.2  — ingest
train, test = partition(samples, training_80)  # 6.1  — split
```

The cell below defines (a copy of) `training_80` and `partition` from Lecture 6.1, then runs them on the output of all four readers — and shows that the split is *byte-identical* across formats.

In [ ]:
# Re-define training_80 and partition exactly as in Lecture 6.1 §8.1 / §8.2.
from typing import Callable, Iterable


def training_80(sample, index: int) -> bool:
    """Lecture 6.1 §8.1 — 80/20 split rule."""
    return index % 5 != 0


def partition(
    samples: Iterable,
    rule: Callable[[object, int], bool],
):
    """Lecture 6.1 §8.2 — higher-order two-pass partition."""
    training = [s for i, s in enumerate(samples) if rule(s, i)]
    test     = [s for i, s in enumerate(samples) if not rule(s, i)]
    return training, test

In [ ]:
# Quick check — pipe every reader's output through the SAME partition.
readers = {
    "CSVIrisReader":        CSVIrisReader(Path("iris.csv")),
    "HeaderlessCSVReader":  HeaderlessCSVReader(Path("iris_headerless.csv")),
    "JSONIrisReader":       JSONIrisReader(Path("iris.json")),
    "NDJSONIrisReader":     NDJSONIrisReader(Path("iris.ndjson")),
}

splits = {}
for name, reader in readers.items():
    samples = list(reader.data_iter())
    train, test = partition(samples, training_80)
    # Compare on species sequences (CSV vs JSON differ in numeric *types* but not in content)
    splits[name] = (
        [s["species"] for s in train],
        [s["species"] for s in test],
    )
    print(f"{name:20s} → {len(train)} train / {len(test)} test")

# All four readers must agree on which species end up in train vs test.
reference = splits["NDJSONIrisReader"]
for name, sp in splits.items():
    assert sp == reference, f"{name} disagrees with NDJSONIrisReader"

print("\nOK: all 4 readers produced the SAME train/test split via the same partition().")
print(f"\ntrain species: {reference[0]}")
print(f"test  species: {reference[1]}")
print("Section 9.7 passed.")

## 10. Summary (Slide 59)

> *"Serialization and File Paths bridge objects to the physical world. Using `pathlib` for locations, and JSON or CSV with intelligent iterators for serialization, we can safely and efficiently move complex objects in and out of flat data storage."*

### What you practiced

| Slide   | Topic                                        | What you built                                                       |
| ------- | -------------------------------------------- | -------------------------------------------------------------------- |
| 5       | String querying                              | `count`, `find`, `rfind`, `startswith`                               |
| 7       | split / join / partition                     | tokenizing & re-joining a sentence                                   |
| 9       | f-string interpolation                       | `f"Hello {name}, …"`                                                 |
| 11      | Escaped braces                               | Java code generator with `{{` / `}}`                                 |
| 13      | Inline expressions + `=` debug               | `f"{name.capitalize()}"`, `f"{a*b=}"`                                |
| 15      | Float precision                              | `f"{x:.2f}"` receipt                                                 |
| 17      | Padding / center / grouping                  | `0>4d`, `*^20`, `,d`                                                 |
| 19      | `__format__` dunder                          | `Temperature` class with `:F` spec                                   |
| 21      | `.format()` delayed templates                | external email template + dict injection                             |
| 23      | Unicode code points                          | `ord(c)` + `:04X` inspection                                         |
| 26      | bytes → str                                  | `b'clich\xc3\xa9'.decode('utf-8')`                                  |
| 28      | encode errors fallback                       | `replace` / `ignore` / `xmlcharrefreplace`                           |
| 30      | `bytearray` mutation                         | in-place index assignment + `append`                                 |
| 32      | pathlib `/` operator                         | `Path.cwd().parent / "ch_02" / "data"`                               |
| 34      | glob / with_suffix / relative_to             | recursive `.txt` → `.bak` mapping                                    |
| 36      | pickle dump / load                           | round-trip a nested list                                             |
| 38      | `__getstate__` / `__setstate__`              | `URLReader` skipping ephemeral state                                 |
| 40      | Schema evolution                             | `Contact` V1 → V2 migration in `__setstate__`                        |
| 42      | JSON dumps / loads                           | indent-2 round-trip                                                  |
| 44      | Custom `JSONEncoder` + `object_hook`         | `Contact` ↔ JSON                                                     |
| 46–58   | **K-NN ingestion pipeline (continuity)**     | `IrisReader` ABC + 4 concrete readers + jsonschema validator + bridge to Lecture 6.1's `partition` |

### Cleanup

In [ ]:
# Tidy up every file we wrote during this notebook.
from pathlib import Path
for name in ("iris.csv", "iris_headerless.csv", "iris.json",
             "iris.ndjson", "iris_mixed.ndjson", "saved_data.pkl"):
    Path(name).unlink(missing_ok=True)
print("Cleaned up: iris.csv, iris_headerless.csv, iris.json, iris.ndjson, iris_mixed.ndjson, saved_data.pkl")

Great work! 🎯 You've now built the **data backbone** for the K-NN classifier — and seen how every byte-level OOP primitive in Python (formatting, encoding, paths, pickle, JSON) folds into a single elegant ingestion pipeline.